In [ ]:
from geopy.geocoders import Nominatim
import pandas as pd
import pickle as pl 
from geopy.exc import GeocoderTimedOut
import time
import re
import googlemaps
from datetime import timedelta
import numpy as np

Este notebook contiene el código usado para calcular la lejania del centro de internamiento al municipio de residencia del interno. 

In [33]:
#df = pd.read_excel('/Users/silvanaruizmedina/Desktop/TFM/Eventos/DATOS/datos 0524/Consulta_suicidios_presentes_090524.xlsx')
df = pd.read_excel('/Users/silvanaruizmedina/Desktop/TFM/Eventos/DATOS/datos0524/Consulta_suicidios_presentes_090524.xlsx')


In [36]:
p_df =df['NOMBRE_CENTRO'].unique().tolist()

In [37]:
a_df = df['PROVINCIA_RESIDENCIA'].unique().tolist()


In [38]:
p_m = df[['NOMBRE_CENTRO', 'PROVINCIA_RESIDENCIA']]

Obtener las coordenadas de los municipios de residencia

In [ ]:

# Inicializar el geolocalizador
geolocator = Nominatim(user_agent="municipio_locator", timeout=20)

# Lista de nombres de cárceles en España
municipio = ['Málaga',
 'Jaén',
 'Granada',
 'Vizcaya',
 'Baleares',
 'Córdoba',
 'Alicante',
 'La Rioja',
 'S. C. Tenerife',
 'Valencia',
 'León',
 'Madrid',
 'Álava',
 'Pontevedra',
 'Valladolid',
 'Ciudad Real',
 'Zaragoza',
 'Cádiz',
 'A Coruña',
 'Cantabria',
 'Sevilla',
 'Melilla',
 'Palencia',
 'Desconocida',
 'Almería',
 'Ourense',
 'Ceuta',
 'Murcia',
 'Toledo',
 'Albacete',
 'Asturias',
 'Las Palmas',
 'Badajoz',
 'Burgos',
 'Salamanca',
 'Cáceres',
 'Castellón',
 'Teruel',
 'Huesca',
 'Huelva',
 'Lugo',
 'Barcelona',
 'Navarra',
 'Guipúzcoa',
 'Cuenca',
 'Segovia',
 'Guadalajara',
 'Zamora',
 'Soria',
 'Ávila',
 'Lleida',
 'Tarragona',
 'No aplica',
 'Girona',
 ]

info_municipio=[]

for m in municipio:
 
    try:
        # Intentar primero usar la dirección completa
        location = geolocator.geocode(m, country_codes='ES')
      
        if location:
            info_municipio.append((m, location.address, location.latitude, location.longitude))
        else:
            print(f"No se encontró una ubicación para {m}")
    except Exception as e:
        print(f"No se pudo obtener la geolocalización para {m: {e}}")
    
    # Esperar para evitar sobrecargar el servidor
    time.sleep(1)

No se encontró una ubicación para No aplica


In [ ]:
# Guardamos para usarlo más tarde
info_municipio
with open('municipio_adresses_nuevo.pickle', 'wb') as f:
    pl.dump(info_municipio,f)

Obtener coordenadas de los centros de internamiento

In [ ]:
# Inicializar el geolocalizador
geolocator = Nominatim(user_agent="prison_locator", timeout=20)

# Lista de nombres de cárceles en España
prisons =["Psiquiatrico Alicante", 
    "Albacete", 
    "Albolote", 
    "Alcala de Guadaira", 
    "Alcazar de San Juan", 
    "Algeciras", 
    "Alicante Cumplimiento", 
    "Almería", 
    "Araba/Álava", 
    "Arrecife de Lanzarote", 
    "Ávila", 
    "Badajoz", 
    "Bilbao (Basauri)", 
    "Burgos", 
    "Cáceres", 
    "Castellón I", 
    "Castellón II (Albocasser)", 
    "Ceuta", 
    "Córdoba", 
    "Cuenca", 
    "Teixeiro (A Coruña)", 
    "Daroca (Zaragoza)", 
    "Dueñas - la Moraleja (Palencia)", 
    "El Dueso (Santoña)", 
    "Herrera de la Mancha", 
    "Huelva", 
    "Ibiza", 
    "Jaén", 
    "A Lama (Pontevedra)", 
    "Las Palmas", 
    "Logroño", 
    "Lugo -Bonxe-", 
    "Madrid I  (A. Henares)", 
    "Madrid III (Valdemoro)", 
    "Madrid IV (Navalcarnero)", 
    "Madrid V (Soto del Real)", 
    "Madrid VI (Aranjuez)", 
    "Madrid VII (Estremera)", 
    "Málaga", 
    "Málaga II", 
    "León", 
    "Melilla", 
    "Murcia", 
    "Orense", 
    "Mallorca", 
    "Pamplona I", 
    "San Sebastian", 
    "Santa Cruz de la Palma", 
    "Segovia", 
    "Sevilla", 
    "Sevilla II (Moron de la Frontera)", 
    "Soria", 
    "Teruel", 
    "Topas (Salamanca)", 
    "Valencia Antoni Asunción Hernández", 
    "Valladolid", 
    "Asturias", 
    "Alicante II (Villena)", 
    "Zaragoza (Zuera)", 
    "Las Palmas II", 
    "Lugo -Monterroso-", 
    "Madrid II  (A. Henares)", 
    "Menorca", 
    'Murcia II',
    'Ocaña I',
    'Ocaña II',
    "Puerto I -Cádiz-", 
    "Puerto II -Cádiz-", 
    "Puerto III -Cádiz-", 
    "Tenerife (El Rosario)", 
    "C.I.S. Carmela Arias y Díaz de Rábago - A Coruña-", 
    "C.I.S. David Beltrán Catalá - Huelva", 
    "C.I.S. Evaristo Martin Nieto (Málaga)", 
    "C.I.S. Guillermo Miranda (Murcia)", 
    "C.I.S. Joaquin Ruiz Jimenez (Mallorca)", 
    "C.I.S. Josefina Aldecoa (Navalcarnero)", 
    "C.I.S. Luis Jimenez de Asua (Sevilla)", 
    "C.I.S. Manuel Montesinos, Algeciras (Cadiz)", 
    "C.I.S. Matilde Cantos Fernandez (Granada)", 
    "C.I.S. Melchor Rodriguez Garcia (Alcalá de Henares)", 
    "C.I.S. Mercedes Pinto (Santa Cruz de Tenerife)", 
    "C.I.S. Torre Espioca, Picassent (Valencia)", 
    "C.I.S Victoria Kent (Madrid)", 
    "Hospital Psiq.Penitenc.Sevilla"
]


# Lista para almacenar las direcciones
addresses = [
    'Otros Partida de Fontcalent S/N 03071 Alicante/Alacant (Alacant/Alicante)',
    'Carrera Ctra. de Ayora Km 2 02006 Albacete (Albacete)',
    'Carretera Colomera Km 6 18220 Albolote (Granada)',
    'Carretera De las canteras S/N 41500 Alcalá de Guadaíra (Sevilla)',
    'Avenida De Quero 51 13600 Alcázar de San Juan (Ciudad Real)',
    'Carretera del cobre 4.5 11206 Algeciras (Cádiz)',
    'Polígono De la Vallonga S/N 03113 Alicante/Alacant (Alacant/Alicante)',
    'Carretera Cueva de los Úbeda Km 2,5 04131 Almería (Almería)',
    'Otros Portillo de San Miguel 1 01230 Iruña Oka/Iruña de Oca (Araba/Álava)',
    'Calle Rafael Alberti 182 35500 Teguise (Palmas, Las)',
    'Carretera De Vicolozano S/N 05194 Ávila (Ávila)',
    'Carretera Olivenza S/N 06011 Badajoz (Badajoz)',
    'Calle Lehendakari Aguirre 92 48970 Basauri (Bizkaia)',
    'Avenida Costa Rica S/N 09071 Burgos (Burgos)',
    'Calle Arroyo Valhondo 1 10004 Cáceres (Cáceres)',
    'Carrera De Alcora Km 10 12006 Castelló de la Plana (Castelló/Castellón)',
    'Carretera Ctra. CV-129 Km 15 12140 Albocàsser (Castelló/Castellón)',
    'Otros Los Rosales s/n 51002 Ceuta (Ceuta)',
    'Otros Autovía de Madrid-Cadiz S/N 14071 Córdoba (Córdoba)',
    'Carretera Tarancón Km 78 16003 Cuenca (Cuenca)',
    'Estrada De Paradela S/N 15310 Curtis (Coruña, A)',
    'Carretera Nombrevilla S/N 50360 Daroca (Zaragoza)',
    'Carretera Local P-120 s/n 34210 Dueñas (Palencia)',
    'Carretera Berria S/N 39740 Santoña (Cantabria)',
    'Carretera De Argamasilla S/N 13200 Manzanares (Ciudad Real)',
    'Carretera de la Ribera S/N 21007 Huelva (Huelva)',
    'Barrio Can Fita S/N 07800 Eivissa (Illes Balears)',
    'Carretera De Bailén-Motril S/N 23009 Jaén (Jaén)',
    'Otros Monte Racelo s/n 36830 Lama, A (Pontevedra)',
    'Otros Salto del Negro S/N 35017 Palmas de Gran Canaria, Las (Palmas, Las)',
    'Camino Calleja Vieja 200 26006 Logroño (Rioja, La)',
    'Carretera Término de Bonxe Km 14 27150 Outeiro de Rei (Lugo)',
    'Carretera Alcalá-Meco Km 5 28803 Alcalá de Henares (Madrid)',
    'Carretera Pinto-San Martín de la Vega Km 5 28340 Valdemoro (Madrid)',
    'Carretera Nacional V 27.7 28600 Navalcarnero (Madrid)',
    'Carretera M-609 Km 3.5 28791 Soto del Real (Madrid)',
    'Carretera Nacional 400, Madrid-Toledo Km 28 28300 Aranjuez (Madrid)',
    'Carretera M-241 Km. 5,750 28595 Estremera (Madrid)',
    'Otros Finca la Moraga S/N 29130 Alhaurín de la Torre (Málaga)',
    'Carretera A Villanueva del Trabuco Km 6 29300 Archidona (Málaga)',
    'Otros Paraje Villahierro S/N 24210 Mansilla de las Mulas (León)',
    'Calle Rio Bidasoa S/N 52002 Melilla (Melilla)',
    'Carretera El Palmar-Mazarrón Km 3 30120 Mazarrón (Murcia)',
    'Carretera De la Derrasa S/N 32710 Pereiro de Aguiar, O (Ourense)',
    'Carretera Ctra. Soller 1 07120 Palma (Illes Balears)',
    'Calle Colina de Santa Lucía S/N 31012 Pamplona/Iruña (Navarra)',
    'Paseo Martutene 1 20014 Donostia/San Sebastián (Gipuzkoa)',
    'Carretera el Galeón 32 38700 Santa Cruz de la Palma (Santa Cruz de Tenerife)',
    'Carretera N-110, Km. 196 s/n 40154 Segovia (Segovia)',
    'Carretera De Torreblanca - Mairena del Alcor S/N 41013 Sevilla (Sevilla)',
    'Carretera Moron-Puebla Cazalla Km. 5,5 41530 Morón de la Frontera (Sevilla)',
    'Calle Calzada Romana S/N 42190 Soria (Soria)',
    'Avenida De Zaragoza S/N 44071 Teruel (Teruel)',
    'Carretera N-630 Km. 313,4 37799 Topas (Salamanca)',
    'S/N 46220 Picassent (València/Valencia)', #
    'Carretera Adanero-Gijón Km. 94 47620 Villanubla (Valladolid)',
    'Calle Villabona S/N 33480 Llanera (Asturias)',
    'Carretera De Madrid a Alicante Km 66 03400 Villena (Alacant/Alicante)',
    'Carretera A-23 S/N 50800 Zuera (Zaragoza)',
    'Carretera General Juan Grande S/N 35107 San Bartolomé de Tirajana (Palmas, Las)',
    'Carretera Vegadeo-Pontevedra S/N 27560 Monterroso (Lugo)',
    'Alcala-Meco Km. 4,5 28805 Alcalá de Henares (Madrid)', 
    'Carrera De Maó a Sant Lluís S/N 07703 Maó (Illes Balears)',
    'Calle Paraje los Charcos s/n 30191 Campos del Río (Murcia)',
    'Calle Mártires de ocaña 4 45300 Ocaña (Toledo)',
    'Calle Mártires de Ocaña 6 45300 Ocaña (Toledo)',
    'Carrera Jerez-Rota Km 6.4 11500 Puerto de Santa María, El (Cádiz)',
    'Carretera Jerez-Rota Km. 5,4 11500 Puerto de Santa María, El (Cádiz)',
    '11500, Cádiz',
    'Camino Escaño S/N 38290 Rosario, El (Santa Cruz de Tenerife)',
    'C/ Alcalde Francisco Vázquez, S/N 15002 A Coruña (A Coruña)',
    'Calle Aruba 2 21007 Huelva (Huelva)',
    'Calle Castelao S/N 29004 Málaga (Málaga)',
    'Carretera Mazarron Km. 3 30120 Murcia (Murcia)',
    'Camí Fondo, 6, Llevant, 07007 Palma, Illes Balears', #cojo una direccion cercana 
    '28600 Navalcarnero, Madrid',
    '41016, Sevilla',
    'Carretera del Cobre, Km 4.5, 11206 Algeciras, Cádiz',
    'Calle Ribera del Beiro S/N 18014 Granada (Granada)',
    'Ctra Alcalá-Meco, km 2, 28805 Alcalá de Henares, Madrid',
    'Calle Ganivet 2 38007 Santa Cruz de Tenerife (Santa Cruz de Tenerife)',
    'S/N 46220 Picassent (València/Valencia)', #
    'Calle Juan de Vera 10 28045 Madrid (Madrid)',
    'Ctra. de Torreblanca - Mairena de Alcor,  41007 Sevilla'

]

prison_addresses= []
def clean_address(address):
    # Reemplazar S/N por ''
    address = re.sub(r"\bS/N\b", '', address)
    # Reemplazar caracteres especiales como '/', '-', 'Otros'
    address = re.sub(r"[./-]", ' ', address)
    # Reemplazar 'Otros' con un espacio vacío
    address = re.sub(r"\bOtros\b", '', address)
    # Eliminar espacios adicionales
    address = re.sub(r'\s+', ' ', address).strip()
    
    
    return address

for prison, address in zip(prisons, addresses):
    clean_prison = clean_address(prison)
    clean_address_str = clean_address(address)
    try:
        # Intentar primero usar la dirección completa
        location = geolocator.geocode(clean_address_str, country_codes='ES')
        
        # Si no se encuentra la ubicación con la dirección, probar con el nombre de la prisión
        if not location:
            location = geolocator.geocode(clean_prison)
         # Si sigue sin encontrar, pero sabemos qué hacer, asignar manualmente
        if not location and prison == "Hospital Psiq.Penitenc.Sevilla": 
            prison_addresses.append((prison, address, 37.3826, -5.9963))
            print(f"Asignadas coordenadas manuales para {prison}")
    
        if location:
            prison_addresses.append((prison, location.address, location.latitude, location.longitude))
        else:
            print(f"No se encontró una ubicación para {clean_prison} con la dirección {clean_address_str}.")
    except Exception as e:
        print(f"No se pudo obtener la geolocalización para {clean_prison}: {e}")
    

    time.sleep(1)





Asignadas coordenadas manuales para Hospital Psiq.Penitenc.Sevilla
No se encontró una ubicación para Hospital Psiq Penitenc Sevilla con la dirección Ctra de Torreblanca Mairena de Alcor, 41007 Sevilla.


In [ ]:
# Guardamos las coordenadas 
with open("prison_adresses_nuevo.pickle", "wb") as f:
    pl.dump(prison_addresses, f)

ME quede AQUIIII Funcion para calcular las distancias 

In [ ]:
def localizacion_centro_provincia(df):
    p_m = df[['NOMBRE_CENTRO', 'PROVINCIA_RESIDENCIA']].copy()

    p_m['NOMBRE_CENTRO_LAT'] = None
    p_m['NOMBRE_CENTRO_LON'] = None
    p_m['PROVINCIA_RESIDENCIA_LAT'] = None
    p_m['PROVINCIA_RESIDENCIA_LON'] = None

    p_m['PROVINCIA_RESIDENCIA'] = p_m['PROVINCIA_RESIDENCIA'].replace(['No se facilita', 'nan', None], 'No aplica')

    p_m['PROVINCIA_RESIDENCIA'] = p_m['PROVINCIA_RESIDENCIA'].fillna('No aplica')
        

    for i, centro in p_m.iterrows():
        for m in prison_addresses:
            if centro['NOMBRE_CENTRO'] == m[0]:
                p_m.loc[i, 'NOMBRE_CENTRO_LAT'] = m[2]
                p_m.loc[i, 'NOMBRE_CENTRO_LON'] = m[3]
            

    for i, provincia in p_m.iterrows():
        if pd.isnull(provincia['PROVINCIA_RESIDENCIA']):
            print(f"No hay provincia en fila {i}")
        else:
            for m in info_municipio:
                if provincia['PROVINCIA_RESIDENCIA'] == m[0]:
                    p_m.loc[i, 'PROVINCIA_RESIDENCIA_LAT'] = m[2]
                    p_m.loc[i, 'PROVINCIA_RESIDENCIA_LON'] = m[3]

    nulos = p_m['PROVINCIA_RESIDENCIA'].isnull().sum()
    print(f"Número total de PROVINCIA_RESIDENCIA vacíos: {nulos}")

    return p_m
p_m = localizacion_centro_provincia(df)   


Número total de PROVINCIA_RESIDENCIA vacíos: 0


En esta sección tenemos las funciones creadas para calcular durante el procesamiento de datos, distancias las cuales no habían sido previamente calculadas. 

In [ ]:
# Función para calcular distancia y tiempo en llegar pro carretera con la API de Google Maps
def calcular_distancia_tiempo(api_key, lat1, lon1, lat2, lon2, max_retries=3):
    try:
        # Inicializar cliente de Google Maps
        gmaps = googlemaps.Client(key=api_key)

        # Coordenadas de origen y destino
        origen = (lat1, lon1)
        destino = (lat2, lon2)

        # Llamada a la API de Distance Matrix
        for intento in range(max_retries):
            try:
                resultado = gmaps.distance_matrix(origen, destino, mode="driving")

                # Verificar si el resultado contiene datos válidos
                if resultado['rows'][0]['elements'][0]['status'] == 'OK':
                    distancia = resultado['rows'][0]['elements'][0]['distance']['text']
                    tiempo = resultado['rows'][0]['elements'][0]['duration']['text']
                else:
                    # Si la API no devuelve un resultado válido, asignar valores predeterminados
                    distancia = "N/A"
                    tiempo = "N/A"

                return distancia, tiempo

            except Exception as e:
                print(f"Error en el intento {intento + 1} de {max_retries}: {e}")
                if intento < max_retries - 1:
                    print(f"Reintentando en 5 segundos...")
                    time.sleep(5)  # Espera 5 segundos antes de reintentar
                else:
                    return "Error", "Error"  # Devuelve error tras exceder el máximo de intentos

    except Exception as e:
        # Maneja errores generales
        print(f"Error al calcular distancia: {e}")
        return "Error", "Error"


# Clave API de google (lo tengo en enviroment secrets por confidencialidad)
api_key = 'API_KEY_GOOGLE'

# Diccionario para almacenar los resultados calculados previamente (caché)
distancia_cache = {}


# Inicializa las columnas para distancia y tiempo
p_m.loc[:, 'DISTANCIA'] = None
p_m.loc[:, 'TIEMPO'] = None

# Tamaño del batch (lote)
batch_size = 53

# Calcular el número total de batches
total_batches = len(p_m) // batch_size + (1 if len(p_m) % batch_size != 0 else 0)

# Iterar sobre las coordenadas y calcular la distancia y el tiempo en lotes
for batch_num, start in enumerate(range(0, len(p_m), batch_size)):
    end = min(start + batch_size, len(p_m))  # Asegurarse de no pasar el final del DataFrame

    # Iterar sobre cada fila en el lote
    for i in range(start, end):
        lat1, lon1 = p_m.loc[i, ['NOMBRE_CENTRO_LAT', 'NOMBRE_CENTRO_LON']]
        lat2, lon2 = p_m.loc[i, ['PROVINCIA_RESIDENCIA_LAT', 'PROVINCIA_RESIDENCIA_LON']]

        # Crear una clave única para las coordenadas (tupla ordenada)
        coordenadas_clave = ((lat1, lon1), (lat2, lon2))

        # Verifica si las coordenadas ya han sido procesadas
        if coordenadas_clave in distancia_cache:
            # Recuperar los valores del caché
            distancia, tiempo = distancia_cache[coordenadas_clave]
            print(f"Usando cache para fila {i + 1} (batch {batch_num + 1}/{total_batches})...")
        else:
            # Verifica si las coordenadas no son NaN
            if pd.notna([lat1, lon1, lat2, lon2]).all():
                print(f"Procesando fila {i + 1} en batch {batch_num + 1}/{total_batches}...")

                # Calcula la distancia y el tiempo usando la función que llama a la API
                distancia, tiempo = calcular_distancia_tiempo(api_key, lat1, lon1, lat2, lon2)

                # Almacenar el resultado en el caché
                distancia_cache[coordenadas_clave] = (distancia, tiempo)
            else:
                # Valores faltantes, no calcular
                print(f"Valores faltantes en la fila {i + 1}, saltando esta fila.")
                distancia, tiempo = "N/A", "N/A"

        # Asignar los valores de distancia y tiempo a las columnas del DataFrame
        p_m.loc[i, 'DISTANCIA'] = distancia
        p_m.loc[i, 'TIEMPO'] = tiempo

# Muestra el DataFrame actualizado
p_m.head()


Procesando fila 1 en batch 1/874...
Procesando fila 2 en batch 1/874...
Usando cache para fila 3 (batch 1/874)...
Procesando fila 4 en batch 1/874...
Procesando fila 5 en batch 1/874...
Procesando fila 6 en batch 1/874...
Procesando fila 7 en batch 1/874...
Procesando fila 8 en batch 1/874...
Procesando fila 9 en batch 1/874...
Procesando fila 10 en batch 1/874...
Procesando fila 11 en batch 1/874...
Procesando fila 12 en batch 1/874...
Valores faltantes en la fila 13, saltando esta fila.
Procesando fila 14 en batch 1/874...
Procesando fila 15 en batch 1/874...
Procesando fila 16 en batch 1/874...
Procesando fila 17 en batch 1/874...
Procesando fila 18 en batch 1/874...
Procesando fila 19 en batch 1/874...
Procesando fila 20 en batch 1/874...
Procesando fila 21 en batch 1/874...
Procesando fila 22 en batch 1/874...
Procesando fila 23 en batch 1/874...
Procesando fila 24 en batch 1/874...
Usando cache para fila 25 (batch 1/874)...
Procesando fila 26 en batch 1/874...
Procesando fila 27 

,NOMBRE_CENTRO,PROVINCIA_RESIDENCIA,NOMBRE_CENTRO_LAT,NOMBRE_CENTRO_LON,PROVINCIA_RESIDENCIA_LAT,PROVINCIA_RESIDENCIA_LON,DISTANCIA,TIEMPO
0,Algeciras,Málaga,36.130506,-5.485195,36.721303,-4.421637,142 km,1 hour 46 mins
1,Zaragoza (Zuera),Jaén,41.865909,-0.788638,37.955728,-3.492056,655 km,6 hours 33 mins
2,Algeciras,Málaga,36.130506,-5.485195,36.721303,-4.421637,142 km,1 hour 46 mins
3,Albolote,Granada,37.305317,-3.684432,37.173499,-3.599534,24.1 km,27 mins
4,Huelva,Málaga,37.307058,-6.921208,36.721303,-4.421637,304 km,3 hours 15 mins


In [ ]:
distancia_cache[coordenadas_clave] = (distancia, tiempo)
p_m.to_csv('Distancias_nuevo.csv')
df_distancias = pd.read_csv('Distancias_nuevo.csv')

In [ ]:
# Creamos el diccionario para guarda las distancias asi cuando más tarde procesemos los datos sea más rápido
distancia_tiempo_dict = {}
for index, row in df_distancias.iterrows():
    key = (row['NOMBRE_CENTRO'], row['PROVINCIA_RESIDENCIA'])
    distancia_tiempo_dict[key] = {
        'DISTANCIA': row['DISTANCIA'],
        'TIEMPO': row['TIEMPO']
    }


{('Algeciras', 'Málaga'): {'DISTANCIA': '142 km', 'TIEMPO': '1 hour 46 mins'},
 ('Zaragoza (Zuera)', 'Jaén'): {'DISTANCIA': '655 km',
  'TIEMPO': '6 hours 33 mins'},
 ('Albolote', 'Granada'): {'DISTANCIA': '24.1 km', 'TIEMPO': '27 mins'},
 ('Huelva', 'Málaga'): {'DISTANCIA': '304 km', 'TIEMPO': '3 hours 15 mins'},
 ('Dueñas - la Moraleja (Palencia)', 'Vizcaya'): {'DISTANCIA': '252 km',
  'TIEMPO': '2 hours 34 mins'},
 ('Mallorca', 'Baleares'): {'DISTANCIA': '25.1 km', 'TIEMPO': '28 mins'},
 ('Córdoba', 'Córdoba'): {'DISTANCIA': '1 m', 'TIEMPO': '1 min'},
 ('Alicante Cumplimiento', 'Alicante'): {'DISTANCIA': '8.6 km',
  'TIEMPO': '14 mins'},
 ('Topas (Salamanca)', 'Vizcaya'): {'DISTANCIA': '394 km',
  'TIEMPO': '3 hours 57 mins'},
 ('Logroño', 'La Rioja'): {'DISTANCIA': '44.4 km', 'TIEMPO': '1 hour 5 mins'},
 ('Tenerife (El Rosario)', 'S. C. Tenerife'): {'DISTANCIA': '2,033 km',
  'TIEMPO': '1 day 16 hours'},
 ('Murcia II', 'No aplica'): {'DISTANCIA': nan, 'TIEMPO': nan},
 ('Araba/Álava

In [56]:
with open('distancia_tiempo_dict_nuevo.pickle', 'bw') as f:
    pl.dump(distancia_tiempo_dict, f)

In [ ]:
df_distancias['TIEMPO']= df_distancias['TIEMPO'].str.replace('mins', 'min')
df_distancias['TIEMPO']= pd.to_timedelta(df_distancias['TIEMPO']).dt.total_seconds()/60
df['LEJANIA_CENTRO_MUNICIPIO'] = df_distancias['TIEMPO']

In [ ]:

# Funcion final usado para el procesamiento de datos (EDA_1)
# En este notebook se hicieron las preubas para comprobar su funcionalidad 
def localizacion_centro_provincia(df):
    # Extraer los centros y provincias del DataFrame
    p_m = df[['NOMBRE_CENTRO', 'PROVINCIA_RESIDENCIA']].copy()

    # Cargar los archivos pickle correctamente
    with open('prison_adresses.pickle', 'rb') as f:
        prison_adress = pl.load(f)

    with open('municipio_adresses.pickle', 'rb') as f:
        municipio_adress = pl.load(f)

    # Inicializar columnas latitud y longitud
    p_m['NOMBRE_CENTRO_LAT'] = None
    p_m['NOMBRE_CENTRO_LON'] = None
    p_m['PROVINCIA_RESIDENCIA_LAT'] = None
    p_m['PROVINCIA_RESIDENCIA_LON'] = None

    # Asignar coordenadas de NOMBRE_CENTRO
    for i, centro in p_m.iterrows():
        for m in prison_adress:
            if centro['NOMBRE_CENTRO'] == m[0]:
                p_m.at[i, 'NOMBRE_CENTRO_LAT'] = m[2]
                p_m.at[i, 'NOMBRE_CENTRO_LON'] = m[3]

    # Asignar coordenadas de PROVINCIA_RESIDENCIA
    for i, provincia in p_m.iterrows():
        for m in municipio_adress:
            if provincia['PROVINCIA_RESIDENCIA'] == m[0]:
                p_m.at[i, 'PROVINCIA_RESIDENCIA_LAT'] = m[2]
                p_m.at[i, 'PROVINCIA_RESIDENCIA_LON'] = m[3]

    # Asegurarse de que las columnas 'DISTANCIA' y 'TIEMPO' existan
    p_m['DISTANCIA'] = None
    p_m['TIEMPO'] = None

    # Función para calcular la distancia y el tiempo usando la API de Google (La funcion anterior la hemos insertado en la funal)
    def calcular_distancia_tiempo(api_key, lat1, lon1, lat2, lon2, max_retries=3):
        try:
            gmaps = googlemaps.Client(key=api_key)
            origen = (lat1, lon1)
            destino = (lat2, lon2)

            for intento in range(max_retries):
                try:
                    resultado = gmaps.distance_matrix(origen, destino, mode="driving")

                    if resultado['rows'][0]['elements'][0]['status'] == 'OK':
                        distancia = resultado['rows'][0]['elements'][0]['distance']['text']
                        tiempo = resultado['rows'][0]['elements'][0]['duration']['text']
                    else:
                        distancia = "N/A"
                        tiempo = "N/A"

                    return distancia, tiempo

                except Exception as e:
                    print(f"Error en el intento {intento + 1} de {max_retries}: {e}")
                    if intento < max_retries - 1:
                        print(f"Reintentando en 5 segundos...")
                        time.sleep(5)
                    else:
                        return "Error", "Error"

        except Exception as e:
            print(f"Error al calcular distancia: {e}")
            return "Error", "Error"

    # Cargar el diccionario de distancias y tiempos
    with open('distancia_tiempo_dict.pickle', 'rb') as f:
        distancia_tiempo_dict = pl.load(f)

    api_key = 'API_KEY_GOOGLE'

    # Tamaño del batch (lote)
    batch_size = 53
    total_batches = len(p_m) // batch_size + (1 if len(p_m) % batch_size != 0 else 0)

    for batch_num, start in enumerate(range(0, len(p_m), batch_size)):
        end = min(start + batch_size, len(p_m))

        for i in range(start, end):
            nombre_centro = p_m.loc[i, 'NOMBRE_CENTRO']
            provincia_residencia = p_m.loc[i, 'PROVINCIA_RESIDENCIA']
            clave = (nombre_centro, provincia_residencia)

            if clave in distancia_tiempo_dict:
                distancia = distancia_tiempo_dict[clave]['DISTANCIA']
                tiempo = distancia_tiempo_dict[clave]['TIEMPO']
                print(f"Usando datos preprocesados para fila {i + 1}...")
            else:
                lat1, lon1 = p_m.loc[i, ['NOMBRE_CENTRO_LAT', 'NOMBRE_CENTRO_LON']]
                lat2, lon2 = p_m.loc[i, ['PROVINCIA_RESIDENCIA_LAT', 'PROVINCIA_RESIDENCIA_LON']]

                if pd.notna([lat1, lon1, lat2, lon2]).all():
                    print(f"Solicitando datos para fila {i + 1}...")
                    distancia, tiempo = calcular_distancia_tiempo(api_key, lat1, lon1, lat2, lon2)
                    distancia_tiempo_dict[clave] = {'DISTANCIA': distancia, 'TIEMPO': tiempo}
                else:
                    distancia, tiempo = np.nan, np.nan

            p_m.at[i, 'DISTANCIA'] = distancia
            p_m.at[i, 'TIEMPO'] = tiempo

    # Convertir el tiempo a minutos
    p_m['TIEMPO'] = p_m['TIEMPO'].str.replace('mins', 'min')
    p_m['TIEMPO'] = pd.to_timedelta(p_m['TIEMPO']).dt.total_seconds() / 60

    # Crear o asignar la columna 'LEJANIA_CENTRO_MUNICIPIO'
    df['LEJANIA_CENTRO_MUNICIPIO'] = p_m['TIEMPO']

    return df


In [ ]:
df = localizacion_centro_provincia(df=df)

Usando datos preprocesados para fila 1...
Usando datos preprocesados para fila 2...
Usando datos preprocesados para fila 3...
Usando datos preprocesados para fila 4...
Usando datos preprocesados para fila 5...
Usando datos preprocesados para fila 6...
Usando datos preprocesados para fila 7...
Usando datos preprocesados para fila 8...
Usando datos preprocesados para fila 9...
Usando datos preprocesados para fila 10...
Usando datos preprocesados para fila 11...
Usando datos preprocesados para fila 12...
Usando datos preprocesados para fila 14...
Usando datos preprocesados para fila 15...
Usando datos preprocesados para fila 16...
Usando datos preprocesados para fila 17...
Usando datos preprocesados para fila 18...
Usando datos preprocesados para fila 19...
Usando datos preprocesados para fila 20...
Usando datos preprocesados para fila 21...
Usando datos preprocesados para fila 22...
Usando datos preprocesados para fila 23...
Usando datos preprocesados para fila 24...
Usando datos preproc